# Normalización del dataset de Smod

### 1. Importar librerias necesarias

In [ ]:
import pandas as pd
import numpy as np
import unicodedata
import os
import csv

from pathlib import Path
from difflib import get_close_matches

### 2. Carga del dataset a normalizar

In [25]:
# Definir la ruta del dataset
dataset_path = r'C:\00 - Proyecto Incendios Galicia 8.0\data\01 - Originales\06 - Smod\06 - Smod.parquet'

# Cargar el archivo CSV directamente
df = pd.read_parquet(dataset_path)

print(f'Filas cargadas: {len(df)}')
print('Columnas disponibles:', list(df.columns))
print(f'Tamaño del dataset: {df.shape[0]} filas x {df.shape[1]} columnas')
display(df.head())

# Mostrar el número de municipios únicos en la columna 'municipio'
print(f"Municipios únicos en el dataset: {df['municipality_name'].nunique()}")

Filas cargadas: 3589391
Columnas disponibles: ['municipality_code', 'municipality_name', 'smod', 'municipio', 'fecha', 'latitud', 'longitud']
Tamaño del dataset: 3589391 filas x 7 columnas


,municipality_code,municipality_name,smod,municipio,fecha,latitud,longitud
0,15001,Abegondo,12,abegondo,1995-01-01,43.227317,-8.288718
1,15001,Abegondo,12,abegondo,1995-01-02,43.227317,-8.288718
2,15001,Abegondo,12,abegondo,1995-01-03,43.227317,-8.288718
3,15001,Abegondo,12,abegondo,1995-01-04,43.227317,-8.288718
4,15001,Abegondo,12,abegondo,1995-01-05,43.227317,-8.288718


Municipios únicos en el dataset: 317


In [ ]:
# Exportar municipios originales a un txt para normalización manual
ruta_txt = r'C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\06 - Smod'

os.makedirs(ruta_txt, exist_ok=True)
archivo_municipios = os.path.join(ruta_txt, 'municipios originales a normalizar.txt')
municipios_originales = sorted(df['municipio'].astype(str).unique())
with open(archivo_municipios, 'w', encoding='utf-8') as f:
    for m in municipios_originales:
        f.write(m + '\n')
print(f"Municipios originales exportados a: {archivo_municipios}")

Municipios originales exportados a: C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\06 - Smod\municipios originales a normalizar.txt


In [27]:
# Mostrar la primera fila como un diccionario columna: valor
primera_fila_dict = df.iloc[0].to_dict()
for k, v in primera_fila_dict.items():
    print(f"{k} = {v}")

municipality_code = 15001
municipality_name = Abegondo
smod = 12
municipio = abegondo
fecha = 1995-01-01 00:00:00
latitud = 43.2273169
longitud = -8.2887175


In [28]:
# Eliminar la columna 'municipality_name'
df.drop(columns=['municipality_name'], inplace=True)

### 2.1 Normalizar los nombres de las columnas

In [29]:
# Mostrar nombres originales de columnas
print('Nombres originales de columnas:')
print(list(df.columns))

# Normalizar nombres de columnas a español, minúsculas y descriptivos según los nombres actuales
columnas_renombrar = {
    'municipality_code': 'codigo_municipio',
    'municipality_name': 'municipio',
    'smod': 'smod',
    'fecha': 'fecha',
    'latitud': 'latitud',
    'longitud': 'longitud',
}

print('\nMapeo de nombres de columnas:')
for k, v in columnas_renombrar.items():
    print(f'{k} -> {v}')

# Renombrar columnas
df.rename(columns=columnas_renombrar, inplace=True)
df.columns = [col.lower() for col in df.columns]

# Eliminar columna de índice si existe
if 'unnamed: 0' in df.columns:
    df.drop(columns=['unnamed: 0'], inplace=True)

print('\nNombres de columnas tras la normalización:')
print(list(df.columns))

Nombres originales de columnas:
['municipality_code', 'smod', 'municipio', 'fecha', 'latitud', 'longitud']

Mapeo de nombres de columnas:
municipality_code -> codigo_municipio
municipality_name -> municipio
smod -> smod
fecha -> fecha
latitud -> latitud
longitud -> longitud

Nombres de columnas tras la normalización:
['codigo_municipio', 'smod', 'municipio', 'fecha', 'latitud', 'longitud']


### 3. Visualización y exploración inicial

In [30]:
# Ver las primeras filas del dataset
df.head()

# Ver información general del dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3589391 entries, 0 to 3589390
Data columns (total 6 columns):
 #   Column            Dtype         
---  ------            -----         
 0   codigo_municipio  int64         
 1   smod              int64         
 2   municipio         object        
 3   fecha             datetime64[ns]
 4   latitud           float64       
 5   longitud          float64       
dtypes: datetime64[ns](1), float64(2), int64(2), object(1)
memory usage: 164.3+ MB


### 4. Cargar el dataset limpio de municipios de Galicia

In [31]:
# Ruta del archivo de municipios limpios
municipios_path = r'C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\01 - municipios\01 - Tabla de municipios.csv'

# Cargar el archivo Excel de municipios
df_municipios = pd.read_csv(municipios_path)

# Mostrar las primeras filas para comprobar que se ha cargado bien
display(df_municipios.head())

# Mostrar el número de municipios únicos en el dataset de municipios
print(f"Municipios únicos en el dataset de municipios: {df_municipios['municipio'].nunique()}")

,municipio,comarca,provincia,altitud,superficie,poblacion,densidad
0,a arnoia,comarca del ribeiro,ourense,76,"20,69",1.000,"48,33"
1,a baña,barcala,a coruña,297,"98,19",3.450,"35,14"
2,a bola,comarca de tierra de celanova,ourense,510,"34,9",1.156,"33,12"
3,a capela,comarca del eume,a coruña,NaN,58,1.232,"21,24"
4,a cañiza,comarca de paradanta,pontevedra,570,"105,04",5.180,"49,31"


Municipios únicos en el dataset de municipios: 315


In [32]:
# Comparar municipios entre el dataset principal y el de referencia
municipios_dataset = set(df['municipio'].dropna().unique())
municipios_referencia = set(df_municipios['municipio'].dropna().unique())

# Municipios en ambos datasets
municipios_comunes = municipios_dataset & municipios_referencia

# Municipios solo en el dataset principal
municipios_solo_dataset = municipios_dataset - municipios_referencia

# Municipios solo en la referencia
municipios_solo_referencia = municipios_referencia - municipios_dataset

print(f"Total municipios en dataset principal: {len(municipios_dataset)}")
print(f"Total municipios en referencia: {len(municipios_referencia)}")
print(f"Municipios coincidentes: {len(municipios_comunes)}")
print(f"Municipios solo en dataset principal: {len(municipios_solo_dataset)}")
if municipios_solo_dataset:
    print('Listado de municipios solo en dataset principal:')
    print(sorted(municipios_solo_dataset))
print(f"Municipios solo en referencia: {len(municipios_solo_referencia)}")
if municipios_solo_referencia:
    print('Listado de municipios solo en referencia:')
    print(sorted(municipios_solo_referencia))

Total municipios en dataset principal: 317
Total municipios en referencia: 315
Municipios coincidentes: 230
Municipios solo en dataset principal: 87
Listado de municipios solo en dataset principal:
['a bana', 'a caniza', 'a coruna', 'a gudina', 'a pobra do brollon', 'a pobra do caraminal', 'a rua', 'abadin', 'arzua', 'as pontes de garcia rodriguez', 'avion', 'banos de molgas', 'barbadas', 'becerrea', 'boboras', 'boqueixon', 'boveda', 'brion', 'cabana de bergantinos', 'calvos de randin', 'camarinas', 'carino', 'castrelo de mino', 'cerdedo cotobade', 'cesuras', 'coiros', 'corcubion', 'dozon', 'dumbria', 'guntin', 'lalin', 'lancara', 'lobeira', 'lourenza', 'malpica de bergantinos', 'manon', 'marin', 'meano', 'melide', 'melon', 'mesia', 'mino', 'moana', 'mondariz balneario', 'mondonedo', 'morana', 'muinos', 'muxia', 'naron', 'negueira de muniz', 'nigran', 'nogueira de ramuin', 'o carballino', 'o paramo', 'o porrino', 'o savinao', 'oimbra', 'oza cesuras', 'oza dos rios', 'padron', 'panton',

### 5. Normalización automática de municipios

In [ ]:
ruta_txt = r'C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\06 - Smod'
os.makedirs(ruta_txt, exist_ok=True)
# Definir las columnas a usar
col_municipio = 'municipio'  # columna a normalizar en el dataset principal
col_ref = 'municipio'        # columna de referencia en el dataset de municipios
df_ref = df_municipios       # referencia oficial

# Función para normalizar nombres eliminando tildes, mayúsculas, signos y espacios extra
def normalizar_nombre(nombre):
    if pd.isnull(nombre):
        return ''
    nombre = str(nombre).strip().lower()
    nombre = ''.join(c for c in unicodedata.normalize('NFD', nombre) if unicodedata.category(c) != 'Mn')
    nombre = nombre.replace('-', ' ').replace(',', '').replace('.', '')
    nombre = ' '.join(nombre.split())
    return nombre

# Diccionario de referencia normalizada para búsqueda rápida
ref_norm = {normalizar_nombre(x): x for x in df_ref[col_ref].dropna().unique()}
ref_norm_keys = set(ref_norm.keys())

# 1. Obtener todos los valores únicos del dataset a normalizar y de la referencia
df[col_municipio] = df[col_municipio].astype(str)
municipios_unicos = set(x for x in df[col_municipio].unique() if isinstance(x, str) and x.strip())
municipios_referencia = set(df_ref[col_ref].dropna().unique())

# 2. Crear mapeo: municipio original -> municipio normalizado (o sugerido, o pendiente)
mapeo = {}
pendientes = []
for m in municipios_unicos:
    clave = normalizar_nombre(m)
    if clave in ref_norm:
        mapeo[m] = ref_norm[clave]
        continue
    sugerencias = get_close_matches(clave, ref_norm_keys, n=1, cutoff=0.8)
    if sugerencias:
        mapeo[m] = ref_norm[sugerencias[0]]
        continue
    # Probar a invertir el orden de las palabras si hay exactamente dos
    partes = clave.split()
    if len(partes) == 2:
        invertido = ' '.join(partes[::-1])
        if invertido in ref_norm:
            mapeo[m] = ref_norm[invertido]
            continue
        sugerencias_inv = get_close_matches(invertido, ref_norm_keys, n=1, cutoff=0.8)
        if sugerencias_inv:
            mapeo[m] = ref_norm[sugerencias_inv[0]]
            continue
    # Si no se encuentra nada, dejar el original y marcar como pendiente
    mapeo[m] = m
    pendientes.append(m)

# 3. Aplicar el mapeo a todo el dataset
df['Municipio_normalizado'] = df[col_municipio].map(mapeo)

# 4. Diagnóstico de diferencias entre dataset y referencia
municipios_normalizados = set(df['Municipio_normalizado'].dropna().unique())
faltan_en_dataset = municipios_referencia - municipios_normalizados
sobran_en_dataset = municipios_normalizados - municipios_referencia

print(f"Municipios únicos en el dataset de referencia: {len(municipios_referencia)}")
print(f"Municipios únicos normalizados en el dataset principal: {len(municipios_normalizados)}")
if faltan_en_dataset:
    print(f"Municipios de la referencia que NO aparecen en el dataset principal: {faltan_en_dataset}")
else:
    print("Todos los municipios de la referencia están presentes en el dataset principal.")
if sobran_en_dataset:
    print(f"Municipios en el dataset principal que NO están en la referencia: {sobran_en_dataset}")
else:
    print("No hay municipios extra en el dataset principal.")

# 5. Exportar el diccionario de correspondencias y los pendientes

archivo_diccionario = os.path.join(ruta_txt, 'diccionario_normalizacion_final.txt')
with open(archivo_diccionario, 'w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f, delimiter='\t')
    writer.writerow(['original', 'normalizado'])
    for k, v in sorted(mapeo.items()):
        writer.writerow([k, v])

archivo_pendientes = os.path.join(ruta_txt, 'municipios_no_normalizados_final.txt')
with open(archivo_pendientes, 'w', encoding='utf-8') as f:
    for m in pendientes:
        f.write(f'{m}\n')

print(f"Municipios únicos originales en el dataset principal: {len(municipios_unicos)}")
print(f"Municipios normalizados automáticamente: {len(mapeo) - len(pendientes)}")
print(f"Municipios pendientes de normalizar: {len(pendientes)}")
print(f'Diccionario de normalización exportado a {archivo_diccionario}')
print(f'Listado de pendientes exportado a {archivo_pendientes}')

Municipios únicos en el dataset de referencia: 315
Municipios únicos normalizados en el dataset principal: 317
Todos los municipios de la referencia están presentes en el dataset principal.
Municipios en el dataset principal que NO están en la referencia: {'cesuras', 'oza dos rios'}
Municipios únicos originales en el dataset principal: 317
Municipios normalizados automáticamente: 315
Municipios pendientes de normalizar: 2
Diccionario de normalización exportado a C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\06 - Smod\diccionario_normalizacion_final.txt
Listado de pendientes exportado a C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\06 - Smod\municipios_no_normalizados_final.txt


In [38]:
# Normalización manual de municipios fusionados históricos sobre la columna final
df['municipio'] = df['municipio'].replace({'cesuras': 'oza-cesuras', 'oza dos rios': 'oza-cesuras'})
print("Normalización manual aplicada: 'cesuras' y 'oza dos rios' ahora son 'oza-cesuras'.")

# Recalcular el conteo tras la corrección manual
municipios_normalizados = set(df['municipio'].dropna().unique())
faltan_en_dataset = municipios_referencia - municipios_normalizados
print(f"Municipios únicos normalizados tras corrección manual: {len(municipios_normalizados)}")
print(f"Municipios de la referencia que NO aparecen tras corrección manual: {faltan_en_dataset}")
print(f"Quedan por normalizar: {len(faltan_en_dataset)}")

Normalización manual aplicada: 'cesuras' y 'oza dos rios' ahora son 'oza-cesuras'.
Municipios únicos normalizados tras corrección manual: 315
Municipios de la referencia que NO aparecen tras corrección manual: set()
Quedan por normalizar: 0


### 6. Exportar el dataset final con municipios normalizados

In [40]:
# Exportar el dataframe final con municipios normalizados (sin columnas duplicadas)

ruta_export = r'C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\06 - Smod'
os.makedirs(ruta_export, exist_ok=True)
archivo_export = os.path.join(ruta_export, 'smod galicia municipios normalizados.csv')

# Eliminar columnas duplicadas si las hubiera
df = df.loc[:, ~df.columns.duplicated()]

df.to_csv(archivo_export, index=False, encoding='utf-8')
print(f'Dataset final exportado como {archivo_export}')

Dataset final exportado como C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\06 - Smod\smod galicia municipios normalizados.csv


In [41]:
# Comprobación final: número de municipios únicos en el CSV exportado vs referencia

csv_exportado = r'C:\00 - Proyecto Incendios Galicia 8.0\data\02 - Municipio normalizado\06 - Smod\smod galicia municipios normalizados.csv'
df_exportado = pd.read_csv(csv_exportado)

municipios_exportados = set(df_exportado['municipio'].dropna().unique())
print(f"Municipios únicos en el CSV exportado: {len(municipios_exportados)}")
print(f"Municipios únicos en la referencia oficial: {len(municipios_referencia)}")

if municipios_exportados == municipios_referencia:
    print('¡El CSV exportado contiene exactamente los mismos municipios que la referencia oficial!')
else:
    diferencia = municipios_exportados.symmetric_difference(municipios_referencia)
    print(f"Diferencias encontradas: {diferencia}")

Municipios únicos en el CSV exportado: 315
Municipios únicos en la referencia oficial: 315
¡El CSV exportado contiene exactamente los mismos municipios que la referencia oficial!
